# Lab 9A: A coordinator with subagents
**Claude Certified Developer (Foundations) · 30-Hour Program for UST · Day 9 · about 30 minutes**

**Objectives**
- Expose a research corpus as in-process tools
- Define subagents with scoped tools
- Run a coordinator and count parallel spawns
- Check coverage and how conflicting figures are reported

**Before you start:** in Colab, click the key icon on the left, add a secret named `ANTHROPIC_API_KEY`, and turn on notebook access. All data in this lab is synthetic.

In [ ]:
# Setup: run once per session
!pip -q install -U anthropic 
import os, json, time, random, importlib, re
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in your environment"
import anthropic
print("anthropic SDK", anthropic.__version__, "| key loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))

In [ ]:
%%writefile config.py
"""One place for Claude settings (same pattern as Day 1)."""
import os
# Confirm current IDs on the models overview page: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL = os.environ.get("CLAUDE_MODEL", "claude-sonnet-5")
FAST_MODEL = os.environ.get("CLAUDE_FAST_MODEL", "claude-haiku-4-5-20251001")
MAX_TOKENS = 1024
MAX_RETRIES = 2
TIMEOUT_S = 120

In [ ]:
import config
importlib.reload(config)
client = anthropic.Anthropic(max_retries=config.MAX_RETRIES, timeout=config.TIMEOUT_S)

def log_response(resp, label=""):
    kinds = [b.type for b in resp.content]
    print(f"{label} stop_reason={resp.stop_reason} blocks={kinds} "
          f"input_tokens={resp.usage.input_tokens} output_tokens={resp.usage.output_tokens}")
    for b in resp.content:
        if b.type == "text":
            print("   ", b.text[:300].replace("\n", " "))

def text_of(resp):
    return "".join(b.text for b in resp.content if b.type == "text")

print("model:", config.MODEL, "| fast model:", config.FAST_MODEL)

In [ ]:
# Setup: Claude Agent SDK (runs the Claude Code runtime as a library; Python 3.10+)
!pip -q install -U claude-agent-sdk
import os
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY"
from claude_agent_sdk import (query, ClaudeSDKClient, ClaudeAgentOptions, AgentDefinition,
                              HookMatcher, tool, create_sdk_mcp_server,
                              AssistantMessage, ResultMessage, TextBlock, ToolUseBlock)
SDK_MODEL = "sonnet"   # alias or full model ID
print("Agent SDK ready")

In [ ]:
import json

async def run_sdk(prompt, options, show=True):
    """Run one Agent SDK query; print tool calls and the final result. Returns (result_msg, tool_calls)."""
    tool_calls, result = [], None
    async for msg in query(prompt=prompt, options=options):
        if isinstance(msg, AssistantMessage):
            names = []
            for b in msg.content:
                if isinstance(b, ToolUseBlock):
                    names.append(b.name); tool_calls.append((b.name, b.input))
                elif isinstance(b, TextBlock) and show:
                    print("  claude:", b.text[:300].replace("\n", " "))
            if names and show:
                print("  tool calls this turn:", names)
        elif isinstance(msg, ResultMessage):
            result = msg
    if result is not None and show:
        cost = getattr(result, "total_cost_usd", None) or 0
        print(f"[result] session={result.session_id} turns={getattr(result, 'num_turns', '?')} cost=${cost:.4f}")
        print((result.result or "")[:1500])
    return result, tool_calls

def text_result(content):
    """Shape a tool result for SDK MCP tools."""
    return {"content": [{"type": "text", "text": content if isinstance(content, str) else json.dumps(content)}]}

In [ ]:
# Synthetic research corpus: "AI in telecom operations" (all figures invented for training)
CORPUS = [
    {"doc_id": "D1", "subtopic": "network operations", "title": "Operator survey", "published": "2025-11-04",
     "text": "Across surveyed operators, AI-assisted fault triage cut mean time to repair by 23 percent."},
    {"doc_id": "D2", "subtopic": "network operations", "title": "Analyst brief", "published": "2026-04-18",
     "text": "Operators using AI fault triage reported a 31 percent reduction in mean time to repair."},
    {"doc_id": "D3", "subtopic": "customer care", "title": "Contact centre study", "published": "2026-02-10",
     "text": "AI-assisted agents resolved 18 percent more contacts on first touch."},
    {"doc_id": "D4", "subtopic": "field service", "title": "Field ops report", "published": "2025-08-22",
     "text": "Predictive dispatch reduced unnecessary truck rolls by 12 percent."},
    {"doc_id": "D5", "subtopic": "billing and fraud", "title": "Fraud desk notes", "published": "2026-06-30",
     "text": "Models flagged SIM-swap fraud 40 percent faster than rules alone."},
    {"doc_id": "D6", "subtopic": "customer care", "title": "Early pilot memo", "published": "2024-12-01",
     "text": "An early pilot improved first-contact resolution by 9 percent."},
]
def search_corpus_py(query):
    words = [w for w in query.lower().split() if len(w) > 3]
    return [{k: d[k] for k in ("doc_id", "title", "published", "subtopic")}
            for d in CORPUS if any(w in (d["subtopic"] + " " + d["text"]).lower() for w in words)]
def load_document_py(doc_id):
    return next((d for d in CORPUS if d["doc_id"] == doc_id), None)
SUBTOPICS = sorted({d["subtopic"] for d in CORPUS})
print("subtopics:", SUBTOPICS)

## 1. Research tools
Search and load are for the searcher; a narrow `verify_fact` is the only tool the writer gets.

In [ ]:
@tool("search_corpus", "Search the research corpus by keywords. Returns doc_id, title, published date, subtopic.", {"query": str})
async def search_corpus(args): return text_result(search_corpus_py(args["query"]))
@tool("load_document", "Load one document's full text and metadata by doc_id.", {"doc_id": str})
async def load_document(args): return text_result(load_document_py(args["doc_id"]) or {"error": "not found"})
@tool("verify_fact", "Check whether a figure appears in the corpus. Returns supporting doc_ids.", {"claim": str})
async def verify_fact(args):
    nums = [t.strip("%") for t in args["claim"].split() if t.strip("%").isdigit()]
    return text_result({"doc_ids": [d["doc_id"] for d in CORPUS if any(n in d["text"] for n in nums)]})
research = create_sdk_mcp_server("research", tools=[search_corpus, load_document, verify_fact])
T = lambda n: f"mcp__research__{n}"

## 2. Subagents
Current docs call the spawning tool **Agent**; some older material calls it **Task**. We allow both.

In [ ]:
AGENTS = {
  "searcher": AgentDefinition(description="Finds and reads sources for ONE assigned subtopic. Use once per subtopic.",
      prompt="Search the corpus for your subtopic, load relevant documents, and return JSON findings: claim, evidence, doc_id, published.",
      tools=[T("search_corpus"), T("load_document")]),
  "writer": AgentDefinition(description="Writes the final cited overview from findings supplied in its prompt.",
      prompt="Write the overview from the findings you are given. Cite doc_id and date for every claim. Show conflicting figures side by side with their dates.",
      tools=[T("verify_fact")]),
}
COORDINATOR = (f"You coordinate research on AI in telecom operations. Cover EVERY subtopic: {SUBTOPICS}. "
               "Spawn one searcher per subtopic, all in the same turn, then pass ALL findings verbatim to the writer. Do not search yourself.")
options = ClaudeAgentOptions(model=SDK_MODEL, setting_sources=[], permission_mode="dontAsk", system_prompt=COORDINATOR, agents=AGENTS,
                             mcp_servers={"research": research}, allowed_tools=["Agent", "Task", T("search_corpus"), T("load_document"), T("verify_fact")])

## 3. Run and count parallel spawns

In [ ]:
spawns, final = [], None
async for msg in query(prompt="Produce the research overview.", options=options):
    if isinstance(msg, AssistantMessage) and getattr(msg, "parent_tool_use_id", None) is None:
        k = sum(1 for b in msg.content if isinstance(b, ToolUseBlock) and b.name in ("Agent", "Task"))
        if k: spawns.append(k); print(f"coordinator turn spawned {k} subagent(s)")
    elif isinstance(msg, ResultMessage):
        final = msg
print("\nspawns per coordinator turn:", spawns, "| parallel:", any(k > 1 for k in spawns))
print(f"turns={getattr(final, 'num_turns', None)} cost=${getattr(final, 'total_cost_usd', 0) or 0:.4f}\n")
print((final.result or "")[:2500])

## 4. Coverage and conflicts

In [ ]:
report = (final.result or "").lower()
display(pd.DataFrame([{"subtopic": s, "covered": s.split()[0] in report} for s in SUBTOPICS]))
print("both conflicting repair-time figures reported:", "23" in report and "31" in report)

If a subtopic is missing, the fix is in the coordinator's plan, not the searcher. Tighten the coordinator prompt and rerun.

## You should now have
- [ ] Scoped tools per subagent
- [ ] Evidence of parallel spawns in one turn
- [ ] A coverage check and conflict check

## Check yourself
1. What would happen if the writer received no findings in its prompt?
2. Why does the writer get verify_fact but not search?
3. How would you cut cost: which subagent could use a cheaper model?